In [1]:
import json
import csv
from pathlib import Path
from statistics import mean

# -------------------------------------------------
# Auto-detect project root (Agent directory)
# -------------------------------------------------
BASE_DIR = Path(r"C:\Users\cyfij\OneDrive\Desktop\DFRWS 2026\Agent\RQs")


CANDIDATE_CSV = BASE_DIR / "RQ2" / "app_total_columns.csv"

print("CSV path:", CANDIDATE_CSV)
print("Exists:", CANDIDATE_CSV.exists())


print("Using BASE_DIR:", BASE_DIR)
print("Using CSV:", CANDIDATE_CSV)
print("CSV exists:", CANDIDATE_CSV.exists())

# -------------------------------------------------
# MODELS
# -------------------------------------------------
MODELS = {
    "gpt_4o_mini": "GPT-4o-mini",
    "gemini_2_5_pro": "Gemini-2.5-Pro",
    "qwen_2_5_72b": "Qwen2.5-72B",
    "llama_3_1_70b": "LLaMA-3.1-70B-Instruct",
    "llama_3_1_8b": "LLaMA-3.1-8B-Instruct",
    "mixtral_8x22b": "Mixtral-8x22B",
    "mixtral_8x7b": "Mixtral-8x7B",
    "mistral_large": "Mistral-Large",
    "gpt_5_1": "GPT-5.1",
    "gpt_4_1": "GPT-4.1",
    "gpt_3_5_turbo": "GPT-3.5-turbo",
}

# -------------------------------------------------
# Load candidate totals per app
# CSV must contain columns:
# app_code,total_columns
# -------------------------------------------------
def load_candidate_totals(csv_path):
    totals = {}
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            app = row.get("app_code")
            total = row.get("total_columns")
            if app and total:
                totals[app] = int(total)
    return totals


# -------------------------------------------------
# Load scanned columns per app
# From normalized_results/<model>/app_level/app_level.jsonl
# -------------------------------------------------
def load_scanned_cols(app_level_jsonl: Path):
    """
    Reads app_level.jsonl (corpus/app-level format).

    Your format:
        {
            "db_path": "selectedDBs\\A1",
            ...
            "Num_of_source_columns_unique": 29
        }

    Returns:
        { "A1": scanned_columns }
    """

    scanned = {}

    with app_level_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            rec = json.loads(line)

            # ---- Extract app from db_path ----
            db_path = rec.get("db_path", "")
            if not db_path:
                continue

            app = Path(db_path).name  # "A1"

            # ---- Use UNIQUE count for efficiency ----
            n = rec.get("Num_of_source_columns_unique")
            if n is None:
                n = rec.get("Num_of_source_columns", 0)

            try:
                n = int(n)
            except:
                n = 0

            # Keep maximum seen per app
            scanned[app] = max(scanned.get(app, 0), n)

    return scanned




# -------------------------------------------------
# Main computation
# -------------------------------------------------
candidate_totals = load_candidate_totals(CANDIDATE_CSV)

results = []

for slug, display_name in MODELS.items():

    app_jsonl = (
        BASE_DIR
        / "normalized_results"
        / slug
        / "app_level"
        / "app_level.jsonl"
    )

    if not app_jsonl.exists():
        results.append((display_name, "xx", "xx"))
        continue

    scanned_cols = load_scanned_cols(app_jsonl)

    examined_vals = []
    reduction_vals = []

    for app, total_cols in candidate_totals.items():

        scanned = scanned_cols.get(app, 0)
        examined_vals.append(scanned)

        if total_cols > 0:
            reduction = 1 - (scanned / total_cols)
            reduction_vals.append(reduction)

    if examined_vals and reduction_vals:
        avg_examined = round(mean(examined_vals), 1)
        avg_reduction = round(mean(reduction_vals) * 100, 1)
        results.append((display_name, avg_examined, f"{avg_reduction}\\%"))
    else:
        results.append((display_name, "xx", "xx"))

# -------------------------------------------------
# Print LaTeX table
# -------------------------------------------------
print(r"\begin{tabular}{|l|p{1.4cm}|p{1.8cm}|}")
print(r"\hline")
print(r"\textbf{Method/LLM} &")
print(r"\textbf{Avg. Cols Examined} &")
print(r"\textbf{Avg. Search Space Reduc.} \\")
print(r"\hline")
print(r"bulk\_extractor-v1.6 (baseline) & NA & 0.0\% \\")
print(r"\hline")

for name, cols, reduc in results:
    print(f"{name} & {cols} & {reduc} \\\\")
    print(r"\hline")

print(r"\end{tabular}")


CSV path: C:\Users\cyfij\OneDrive\Desktop\DFRWS 2026\Agent\RQs\RQ2\app_total_columns.csv
Exists: True
Using BASE_DIR: C:\Users\cyfij\OneDrive\Desktop\DFRWS 2026\Agent\RQs
Using CSV: C:\Users\cyfij\OneDrive\Desktop\DFRWS 2026\Agent\RQs\RQ2\app_total_columns.csv
CSV exists: True
\begin{tabular}{|l|p{1.4cm}|p{1.8cm}|}
\hline
\textbf{Method/LLM} &
\textbf{Avg. Cols Examined} &
\textbf{Avg. Search Space Reduc.} \\
\hline
bulk\_extractor-v1.6 (baseline) & NA & 0.0\% \\
\hline
GPT-4o-mini & 9.1 & 85.0\% \\
\hline
Gemini-2.5-Pro & 0 & 100.0\% \\
\hline
Qwen2.5-72B & 31.5 & 41.2\% \\
\hline
LLaMA-3.1-70B-Instruct & 1 & 97.5\% \\
\hline
LLaMA-3.1-8B-Instruct & 0.4 & 99.9\% \\
\hline
Mixtral-8x22B & 3.2 & 90.3\% \\
\hline
Mixtral-8x7B & 2.6 & 97.2\% \\
\hline
Mistral-Large & 24.9 & 75.6\% \\
\hline
GPT-5.1 & 29.9 & 81.0\% \\
\hline
GPT-4.1 & 20.5 & 69.7\% \\
\hline
GPT-3.5-turbo & 2.8 & 98.6\% \\
\hline
\end{tabular}
